# Étape 3 — Analyse des résultats (Régression)
**Prérequis** : avoir exécuté `ComparaisonReg_final.ipynb` (FICHIERS FINAUX/) puis sauvegardé PREV :
```python
# A exécuter à la fin de ComparaisonReg_final.ipynb :
PREV.to_csv("PREV.csv", index=False)
```

Ce notebook :
- Calcule la **MSE par méthode** (erreur quadratique moyenne de validation croisée)
- Classe les méthodes de la moins erreur à la plus grande
- Visualise la comparaison
- Analyse la stabilité des λ optimaux Lasso par fold (si disponible)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ← Modifier si PREV.csv est ailleurs
DOSSIER_PREV = "../FICHIERS FINAUX/"

# Metrique de comparaison : 'mse' ou 'rmse' ou 'mae'
METRIQUE = "mse"

## 1. Chargement de PREV.csv

In [ ]:
import os
chemin_prev = DOSSIER_PREV + "PREV.csv"

if not os.path.exists(chemin_prev):
    raise FileNotFoundError(
        f"{chemin_prev} introuvable.\n"
        "Etapes :\n"
        "  1. Ouvrir FICHIERS FINAUX/ComparaisonReg_final.ipynb\n"
        "  2. Executer toutes les cellules\n"
        "  3. Ajouter et executer : PREV.to_csv('PREV.csv', index=False)"
    )

PREV = pd.read_csv(chemin_prev)
print(f"PREV charge : {PREV.shape[0]} observations x {PREV.shape[1]} colonnes")
print(f"Methodes disponibles : {list(PREV.columns[2:])}")
print(f"Blocs CV : {PREV['bloc'].nunique()}")
PREV.head(3)

## 2. Calcul des métriques par méthode

In [ ]:
# Colonnes des methodes (tout sauf 'bloc' et 'Y')
methodes = [c for c in PREV.columns if c not in ["bloc", "Y"]]

# Calcul des residus
residus = PREV[methodes].sub(PREV["Y"], axis=0)

metriques = pd.DataFrame(index=methodes)
metriques["MSE"]   = (residus ** 2).mean().round(2)
metriques["RMSE"]  = np.sqrt((residus ** 2).mean()).round(3)
metriques["MAE"]   = residus.abs().mean().round(2)

# Rang (1 = meilleure methode)
metriques["Rang_MSE"] = metriques["MSE"].rank().astype(int)

metriques_tries = metriques.sort_values("MSE")
print("Comparaison des methodes (triee par MSE croissante) :")
print(metriques_tries.to_string())

## 3. Visualisation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Graphe 1 : MSE par methode (barres horizontales)
m = metriques_tries
colors = ["steelblue" if mse == m.MSE.min() else "lightsteelblue" for mse in m.MSE]
ax1.barh(m.index, m.MSE, color=colors)
ax1.set_xlabel("MSE (validation croisee)")
ax1.set_title("MSE par methode")
ax1.axvline(x=m.MSE.min(), color="orange", linestyle="--", linewidth=1)

# Graphe 2 : Distribution des residus par methode (boxplot)
residus_melted = residus[methodes].melt(var_name="Methode", value_name="Residu")
methodes_triees = metriques_tries.index.tolist()
residus_grouped = [residus[m].values for m in methodes_triees]
ax2.boxplot(residus_grouped, labels=methodes_triees, vert=False)
ax2.set_xlabel("Residu (Y - Y_predit)")
ax2.set_title("Distribution des residus par methode")
ax2.axvline(x=0, color="grey", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.show()

## 4. Stabilité des λ Lasso par fold
Un lambda très variable d'un fold à l'autre indique une instabilité du modèle.  
Charger `lambdaoptlasso.csv` si vous l'avez sauvegardé depuis `ComparaisonReg_final.ipynb` :
```python
# A la fin de ComparaisonReg_final.ipynb :
pd.Series(lambdaoptlasso).to_csv("lambdaoptlasso.csv", index=False)
```

In [ ]:
chemin_lambda = DOSSIER_PREV + "lambdaoptlasso.csv"
if os.path.exists(chemin_lambda):
    lambdas = pd.read_csv(chemin_lambda, header=None, names=["lambda"])
    print(f"Lambda Lasso par fold :")
    print(lambdas.T.to_string(index=False))
    print(f"Moyenne : {lambdas['lambda'].mean():.4f}")
    print(f"Ecart-type : {lambdas['lambda'].std():.4f}")
    if lambdas['lambda'].std() / lambdas['lambda'].mean() > 0.3:
        print("=> Lambda tres variable : le modele Lasso est instable sur ces donnees.")
    else:
        print("=> Lambda stable : bonne reproductibilite du modele Lasso.")
else:
    print(f"{chemin_lambda} non trouve — sauvegarder lambdaoptlasso depuis ComparaisonReg_final.ipynb")

## 5. Conclusion
Completer apres execution :
- **Meilleure methode** : ... (MSE = ...)
- **Jeu de features utilise** : X / XI / XP / XIP (modifier en haut de ComparaisonReg_final.ipynb)
- **Ecart avec MCO** : ...

**Pistes d'amelioration** :
- Reactiver BIC/AIC (commentes dans ComparaisonReg_final.ipynb) si le nb de features est raisonnable
- Tester `X = XIP` (interactions + polynomes) si MSE encore elevee
- Augmenter le nb de composantes dans GridSearch PCR/PLS si nbaxes=20 est atteint
- Ajuster `min_samples_leaf` sur l'arbre de decision